# Gene copy number feature extraction

Similar to the method used for processing gene mutation data, we also employed the Chi-square-G test to handle gene copy number data, with the following specific steps: First, we converted the copy number ratio to the absolute copy number using the formula absolute copy number=ploidy*(copy number ratio), and rounded the result to the nearest integer. Here, ploidy refers to the chromosomal copy number in a cell, reflecting the copy number of a specific chromosome [76]. It is well established that the normal gene copy number on human chromosomes is 2, with values greater than 2 indicating gene amplification and values less than 2 denoting gene deletion. Therefore, we recorded genes with an absolute copy number greater than or equal to 3 as amplified, those less than or equal to 1 as deleted, and those equal to 2 as normal. Subsequently, we counted the number of genes exhibiting these three types of copy number alterations within and outside the given gene set for each cell line, constructing a contingency table similar to Table 5. Finally, we employed the Chi-square-G test method to calculate the deviation between the observed and expected frequencies based on the contingency table, obtaining the chi-square statistic. The chi-square statistic represents the overall deviation between the two groups, with a larger value indicating a more pronounced difference in copy number variations between genes within and outside of the gene set. Ultimately, we utilized this chi-square statistic as the gene copy number feature for cell lines.

![Table 5. Copy number contingency table](img/img_1.png)

In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats.contingency import expected_freq
from scipy.stats import power_divergence

In [2]:
def remove_brackets(x):
    return x.split('(')[0].strip()

In [3]:
df_gene_cn = pd.read_csv('data/OmicsCNGene23Q2.csv',index_col=0)
df_gene_cn.columns = df_gene_cn.columns.map(remove_brackets)
print(df_gene_cn.shape)

(1814, 25368)


In [4]:
df_exp = pd.read_csv('raw_data/OmicsExpressionProteinCodingGenesTPMLogp1-23Q2.csv')
df_exp.columns = df_exp.columns.map(remove_brackets)
print(df_exp.shape)

(1450, 19194)


In [5]:
genes_exp = df_exp.columns[1:]
df_gene_cn = df_gene_cn.loc[:, df_gene_cn.columns.isin(genes_exp)]
print(df_gene_cn.shape)
# df_gene_cn.head(5)

(1814, 19180)


In [6]:
# Process header data
# Extract the gene names from the header of the copy number data and convert them into a set.
df_gene_cn_genes = df_gene_cn.iloc[:,1:].columns.values
# Remove the parentheses and their contents from the header elements, and also remove spaces.
# df_gene_cn.columns = [re.sub(r'\([^)]*\)', '', gene).strip() 
#                       for gene in df_gene_cn.iloc[:,0:].columns.values]
df_gene_cn_genes = set(df_gene_cn.iloc[:,1:].columns)
# print(mutation_genes)
print(len(df_gene_cn_genes))
print(df_gene_cn.columns)
print(type(df_gene_cn_genes))

df_cell_names = df_gene_cn.index
print(df_cell_names)

19179
Index(['OR4F5', 'OR4F29', 'OR4F16', 'SAMD11', 'NOC2L', 'KLHL17', 'PLEKHN1',
       'PERM1', 'HES4', 'ISG15',
       ...
       'BPY2', 'DAZ1', 'DAZ2', 'CDY1B', 'BPY2B', 'DAZ3', 'DAZ4', 'BPY2C',
       'CDY1', 'POLR2J3'],
      dtype='object', length=19180)
<class 'set'>
Index(['ACH-000667', 'ACH-001148', 'ACH-000159', 'ACH-001675', 'ACH-000947',
       'ACH-000120', 'ACH-000382', 'ACH-000594', 'ACH-000865', 'ACH-000008',
       ...
       'ACH-002378', 'ACH-001224', 'ACH-000299', 'ACH-001047', 'ACH-000731',
       'ACH-000185', 'ACH-001044', 'ACH-000494', 'ACH-001087', 'ACH-000526'],
      dtype='object', length=1814)


In [7]:
df_gene_set = pd.read_csv('data/c2.cp.kegg_legacy.v2023.2.Hs.symbols.csv',header=None)
print(df_gene_set.shape)
# df_gene_set.head(5)

(186, 390)


In [8]:
# Processed pathway data.
pathways = []
for i,data in enumerate(df_gene_set.values):

    pathway_name = data[0]
    
    pathway_genes = data[1:]
    lst = [pathway_name,pathway_genes]
    pathways.append(lst)
#     print(gene_set,type(gene_set))
#     print(pathway_genes)
#     break
# print(pathways)

df_pathways = pd.DataFrame(pathways)
# print(df.shape)

print(df_pathways.shape)
df_pathways.head(5)

(186, 2)


,0,1
0,KEGG_ABC_TRANSPORTERS,"[ABCA1, ABCA10, ABCA12, ABCA13, ABCA2, ABCA3, ..."
1,KEGG_ACUTE_MYELOID_LEUKEMIA,"[AKT1, AKT2, AKT3, ARAF, BAD, BRAF, CCNA1, CCN..."
2,KEGG_ADHERENS_JUNCTION,"[ACP1, ACTB, ACTG1, ACTN1, ACTN2, ACTN3, ACTN4..."
3,KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY,"[ACACB, ACSL1, ACSL3, ACSL4, ACSL5, ACSL6, ADI..."
4,KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM,"[ABAT, ACY3, ADSL, ADSS1, ADSS2, AGXT, AGXT2, ..."


In [9]:
# Calculate the absolute copy number for all values in the matrix.
df_gene_cn = 2*(2 ** df_gene_cn.iloc[:,1:] - 1)

# df_gene_cn.head(5)

In [11]:
# Chi-square-G Test
def Chi_square_G_Test(r):
    expected_matrix = expected_freq(r)
    # Check if there are any zeros in the expected_matrix.
    if np.any(expected_matrix == 0):
        # Set the chi-square value directly to 0.
        # print('expected_matrix exist 0')
        return 0
        
    # Check if there are any numbers less than 1 in the matrix.
    if np.any(expected_matrix < 5):
        # Use the G-test for maximum likelihood statistical significance testing, which is approximately equivalent to the chi-square test.
        x2 = power_divergence(r,expected_matrix, ddof=r.size - 2, axis=None,lambda_='log-likelihood')[0]
        return x2
    else:
        # No continuity correction is needed.
        x2 = np.sum((r - expected_matrix) ** 2 / expected_matrix)
    return x2

In [13]:
Copy_number_variability_scores = []
count_zero = 0
for i,data in enumerate(df_pathways.values):
    
    pathway_name = data[0]

    gene_set = set(data[1])

    # If there is no intersection between `mutation_genes` and `df_pathways`, skip this iteration and proceed to the next loop.
    if not df_gene_cn_genes & gene_set: 
        
        df_pathways.drop(index=[i],inplace=True)
        continue
    
    differences = df_gene_cn_genes - gene_set

    intersections = df_gene_cn_genes & gene_set
    
    internal = df_gene_cn[list(intersections)]
    
    external = df_gene_cn[list(differences)]

    internal_lenth = len(intersections)
    
    external_lenth = len(differences)
    
    # Number of genes with amplification or deletion inside the gene set.
    internal_count = ((internal >= 2.5) | (internal < 1.5)).sum(axis=1)  
    # Number of genes with amplification or deletion outside the gene set.
    external_count = ((external >= 2.5) | (external < 1.5)).sum(axis=1) 

    X_square = []
    for inter,exter in zip(internal_count,external_count):
        R = np.array([[inter,internal_lenth- inter],
                      [exter,external_lenth - exter]])

        x2 = Chi_square_G_Test(R)
        if x2 == 0 :

            count_zero = count_zero + 1
            print('Number of zero values generated:',count_zero)
            print(R)
        
        X_square.append(x2)

    
    Copy_number_variability_scores.append(X_square)
    print('pathway_name:',i+1,'.',pathway_name,'Calculation completed!')
    
    
df_Copy_number_variability_scores = pd.DataFrame(Copy_number_variability_scores)
print('Done!')
print(df_Copy_number_variability_scores.shape)

通路： 1 . KEGG_ABC_TRANSPORTERS 计算完成！
通路： 2 . KEGG_ACUTE_MYELOID_LEUKEMIA 计算完成！
通路： 3 . KEGG_ADHERENS_JUNCTION 计算完成！
通路： 4 . KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY 计算完成！
通路： 5 . KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM 计算完成！
通路： 6 . KEGG_ALDOSTERONE_REGULATED_SODIUM_REABSORPTION 计算完成！
通路： 7 . KEGG_ALLOGRAFT_REJECTION 计算完成！
通路： 8 . KEGG_ALPHA_LINOLENIC_ACID_METABOLISM 计算完成！
通路： 9 . KEGG_ALZHEIMERS_DISEASE 计算完成！
通路： 10 . KEGG_AMINOACYL_TRNA_BIOSYNTHESIS 计算完成！
通路： 11 . KEGG_AMINO_SUGAR_AND_NUCLEOTIDE_SUGAR_METABOLISM 计算完成！
通路： 12 . KEGG_AMYOTROPHIC_LATERAL_SCLEROSIS_ALS 计算完成！
通路： 13 . KEGG_ANTIGEN_PROCESSING_AND_PRESENTATION 计算完成！
通路： 14 . KEGG_APOPTOSIS 计算完成！
通路： 15 . KEGG_ARACHIDONIC_ACID_METABOLISM 计算完成！
通路： 16 . KEGG_ARGININE_AND_PROLINE_METABOLISM 计算完成！
通路： 17 . KEGG_ARRHYTHMOGENIC_RIGHT_VENTRICULAR_CARDIOMYOPATHY_ARVC 计算完成！
通路： 18 . KEGG_ASCORBATE_AND_ALDARATE_METABOLISM 计算完成！
通路： 19 . KEGG_ASTHMA 计算完成！
通路： 20 . KEGG_AUTOIMMUNE_THYROID_DISEASE 计算完成！
通路： 21 . KEGG_AXON_GUIDANCE 计算

In [14]:
df_Copy_number_variability_scores.index = df_pathways[0]
df_Copy_number_variability_scores.columns = df_cell_names

In [15]:
df_Copy_number_variability_scores.to_csv('data/CNV_Cardinality_analysis_of_variance_Latest_KEGG186.csv')

In [14]:
# row: pathway ID; column: cell line name sample.
pathways_index = df_pathways.iloc[:, 0]
col_cell_line_names = df_cell_names
df_Cardinality_analysis_of_variance = pd.DataFrame(df_Copy_number_variability_scores,index=pathways_index,columns=col_cell_line_names)

In [ ]:
df_Cardinality_analysis_of_variance.to_csv('Cardinality_analysis_of_variance.csv')